# Conformal Prediction for Multi-Classifier Uncertainty Quantification
## A Comparative Study of Standard, Class-Conditional, and Mondrian Approaches on Balanced Tabular Datasets

---

**Author:** Muhammad Adeel
**Affiliation:** Department of BBIT (Bachelor of Business Information Technology), Virtual University of Pakistan, Lahore
**Email:** ai.adeelv1@gmail.com
**LinkedIn:** https://www.linkedin.com/in/muhammadadeelai/
**GitHub:** https://github.com/adeeljames
**Date:** 2026

---

### Abstract

This notebook accompanies the research paper that systematically evaluates four families of Conformal Prediction (CP) methods - Standard (marginal), Class-Conditional, Mondrian, and Adaptive Prediction Sets (APS) - across three publicly available tabular medical datasets with mild class imbalance. The study compares five base classifiers (Logistic Regression, Random Forest, XGBoost, LightGBM, SVM) and evaluates coverage validity, prediction set efficiency, class-conditional fairness, and calibration across five coverage levels (80%, 85%, 90%, 95%, 99%). All CP methods are implemented from first principles using only NumPy and scikit-learn, providing a transparent and reproducible experimental framework.

**Keywords:** Conformal Prediction, Uncertainty Quantification, Distribution-Free Inference, Class-Conditional Coverage, Mondrian Conformal Prediction, Adaptive Prediction Sets, Mild Imbalance, Tabular Classification

---

### Reproducibility Statement

- **Datasets:** All three datasets are fetched automatically from OpenML via `sklearn.datasets.fetch_openml`. No manual download or upload is required.
- **Hardware:** Google Colab free CPU runtime is sufficient. No GPU needed.
- **Runtime:** Approximately 25-35 minutes for a full run of all 300 main configurations plus cross-dataset transferability and reject-option experiments.
- **Random seed:** All experiments use `random_state=42` for full reproducibility.


# 1. Introduction

## 1.1 Motivation

Modern machine learning classifiers produce point predictions with no built-in guarantee that the true label is actually included in the prediction. In safety-critical applications - medical diagnosis, autonomous driving, credit decisions - this is unacceptable.

**Conformal Prediction (CP)** is a distribution-free, model-agnostic framework that converts any pre-trained classifier's outputs into **prediction sets** with **finite-sample, theoretically guaranteed coverage**. Given a target miscoverage rate alpha in (0, 1), CP guarantees:

    P[Y_test in C_hat(X_test)] >= 1 - alpha

This guarantee holds for **any** underlying classifier, **any** data distribution, and **any** sample size - making CP one of the most powerful yet underused tools in modern ML.

## 1.2 Research Gap

While CP has been studied extensively for **extreme imbalance** (e.g., 0.17% fraud) and **balanced** settings, the behavior of different CP families on **mildly imbalanced tabular datasets** (15%-45% minority class) remains under-explored. This is precisely the regime most common in real medical and business datasets.

Most prior works (Ding et al. NeurIPS 2024; AAAI 2024) propose **new methods**; very few provide **comprehensive empirical comparisons** of the three foundational CP families on identical datasets, classifiers, and metrics.

## 1.3 Contributions

1. **Systematic empirical comparison** of 4 CP families (Standard, Class-Conditional, Mondrian, APS) on 3 mildly imbalanced medical datasets, with 5 base classifiers and 5 coverage levels = **300 main configurations**.
2. **From-scratch implementation** of all CP methods using only NumPy, ensuring full transparency and pedagogical value.
3. **Cross-dataset transferability analysis** - train classifier on one dataset, calibrate on a second, test on a third.
4. **Reject-option integration** study, where CP is used to abstain from low-confidence predictions.
5. **Practitioner decision framework** that translates the empirical findings into actionable guidelines.

## 1.4 BBIT Student Perspective

As a Bachelor of Business Information Technology (BBIT) student at the Virtual University of Pakistan, the author approaches this problem from both **business risk** and **technical implementation** perspectives. Healthcare decision-support systems must balance diagnostic accuracy with regulatory compliance (FDA, EMA), patient safety, and operational efficiency.


# 2. Environment Setup

This cell installs and imports all required libraries. If running on Google Colab, the `pip install` commands will execute silently.

**Required libraries:**
- `numpy`, `pandas` - data handling
- `scikit-learn` - base classifiers, OpenML fetch, metrics
- `xgboost`, `lightgbm` - gradient boosting classifiers
- `matplotlib`, `seaborn` - visualization


In [ ]:
# Install required packages (silent on Colab)
import subprocess
import sys

def install_if_missing(packages):
    for pkg, import_name in packages:
        try:
            __import__(import_name)
        except ImportError:
            print(f"Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

install_if_missing([
    ("xgboost", "xgboost"),
    ("lightgbm", "lightgbm"),
    ("seaborn", "seaborn"),
])

print("All packages ready.")


All packages ready.


In [ ]:
# Core imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import math
import time
import os
from collections import defaultdict

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import xgboost as xgb
import lightgbm as lgb

# Plotting configuration
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_style("whitegrid")

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Create output directory for results
OUTPUT_DIR = '/content/cp_results' if os.path.exists('/content') else './cp_results'
FIGURES_DIR = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")
print(f"Figures directory: {FIGURES_DIR}")

print("=" * 70)
print("Setup complete. Author: Muhammad Adeel (BBIT, VU Pakistan, Lahore)")
print("Email: ai.adeelv1@gmail.com | LinkedIn: muhammadadeelai | GitHub: adeeljames")
print("=" * 70)


Output directory: /content/cp_results
Figures directory: /content/cp_results/figures
Setup complete. Author: Muhammad Adeel (BBIT, VU Pakistan, Lahore)
Email: ai.adeelv1@gmail.com | LinkedIn: muhammadadeelai | GitHub: adeeljames


# 3. Background: Conformal Prediction Theory

## 3.1 Setup

Let (X_1, Y_1), ..., (X_n, Y_n), (X_test, Y_test) be n+1 i.i.d. (or more generally, exchangeable) random variables from a joint distribution P on X x Y, where Y = {1, 2, ..., K} is a finite label set.

Given:
- Training data {(X_i, Y_i)}_{i=1}^{n_train}
- Calibration data {(X_i, Y_i)}_{i=1}^{n_calib}
- Test point X_test and target miscoverage alpha in (0, 1)

We seek a **prediction set** C_hat(X_test) subset of Y such that:

    P[Y_test in C_hat(X_test)] >= 1 - alpha

## 3.2 Nonconformity Score

A **nonconformity score** s(x, y) in R measures how "unusual" the pair (x, y) is relative to the calibration data. Higher scores indicate more atypical pairs.

For classification, we use the **LAC (Least Ambiguous Classifier) score**:

    s(x, y) = 1 - pi_hat_y(x)

where pi_hat_y(x) is the predicted probability of class y for input x from any pre-trained classifier.

## 3.3 Theorem 1 (Standard CP Coverage Guarantee)

**Theorem (Vovk, 2005; Lei et al., 2018).** Suppose the data are exchangeable. Define the (1-alpha) quantile of the calibration scores as:

    q_hat_{1-alpha} = Quantile_{ceil((n+1)(1-alpha)/n)/n}(s_1, ..., s_n)

using the "higher" method. Then the prediction set:

    C_hat(X_test) = { y in Y : s(X_test, y) <= q_hat_{1-alpha} }
                  = { y in Y : pi_hat_y(X_test) >= 1 - q_hat_{1-alpha} }

satisfies the **marginal coverage** guarantee:

    P[Y_test in C_hat(X_test)] >= 1 - alpha

**Proof sketch.** By exchangeability, the rank of s(X_test, Y_test) among {s_1, ..., s_n, s(X_test, Y_test)} is uniformly distributed on {1, 2, ..., n+1}. Hence P[s(X_test, Y_test) <= q_hat_{1-alpha}] >= 1 - alpha. The finite-sample correction ceil((n+1)(1-alpha)/n)/n ensures this lower bound holds exactly, not asymptotically.

## 3.4 Theorem 2 (Class-Conditional CP Coverage)

**Theorem (Vovk, 2005; Sadinle et al., 2019).** For each class k in Y, let q_hat^{(k)}_{1-alpha} be the ceil((n_k+1)(1-alpha)/n_k)/n_k quantile of the calibration scores restricted to class k. Then:

    C_hat(X_test) = { k in Y : pi_hat_k(X_test) >= 1 - q_hat^{(k)}_{1-alpha} }

satisfies the **per-class coverage** guarantee:

    P[Y_test in C_hat(X_test) | Y_test = k] >= 1 - alpha,  for all k in Y

This is stronger than marginal coverage and is essential in imbalanced settings where Standard CP can systematically under-cover minority classes.

## 3.5 Theorem 3 (Mondrian CP Coverage)

**Theorem (Vovk, 2005; Lange & Sachs, 2022).** Let Pi: X x Y -> G be a partition function mapping each sample to one of G disjoint groups G = {g_1, ..., g_G}. For each group g, let q_hat^{(g)}_{1-alpha} be the quantile of the calibration scores restricted to group g. Then:

    C_hat(X_test) = { k in Y : s(X_test, k) <= q_hat^{(Pi(X_test, k))}_{1-alpha} }

satisfies the **group-conditional coverage** guarantee:

    P[Y_test in C_hat(X_test) | Pi(X_test, Y_test) = g] >= 1 - alpha,  for all g in G

When the partition is the label itself (Pi(x, y) = y), Mondrian CP reduces to Class-Conditional CP. Mondrian CP is more general: it can partition by any grouping (e.g., demographic subgroup, severity tier, data source).

## 3.6 Adaptive Prediction Sets (APS)

APS, introduced by Romano et al. (2020), uses a different nonconformity score based on the **cumulative sum of softmax probabilities**. The score for a pair (x, y) is:

    s(x, y) = sum_{j in sigma_x^{-1}(y)} pi_hat_{sigma_x(j)}(x)

where sigma_x is a permutation that sorts classes by descending predicted probability. APS produces **adaptive-size** prediction sets: easy cases get small sets, ambiguous cases get larger sets.


# 4. Datasets

We use **three publicly available tabular medical datasets** from OpenML, chosen to span different sample sizes, feature dimensionalities, and mild (non-extreme) class imbalance ratios.

| # | Dataset | OpenML ID | Rows | Features | Classes | Minority % | Domain |
|---|---------|-----------|------|----------|---------|-----------|--------|
| 1 | Blood Transfusion Service Center | 1464 | 748 | 4 | 2 | 23.80% | Medical (blood donation) |
| 2 | Wisconsin Diagnostic Breast Cancer (WDBC) | 1510 | 569 | 30 | 2 | 37.26% | Medical (oncology) |
| 3 | EEG Eye State | 1471 | 14,980 | 14 | 2 | 44.88% | Medical (neuroscience) |

**Why these datasets?**
- All three are in the **mild imbalance regime** (15-45% minority), which is the most common real-world medical classification scenario.
- All are **publicly accessible** via `sklearn.datasets.fetch_openml` - no manual download or upload required.
- They span a range of **sample sizes** (569 to 14,980) and **feature dimensions** (4 to 30), enabling diverse evaluation.
- All belong to the **medical domain**, providing a coherent narrative for healthcare AI applications.


In [ ]:
# Load all three datasets from OpenML
DATASETS_CONFIG = [
    {'id': 1464, 'name': 'Blood Transfusion', 'short': 'blood'},
    {'id': 1510, 'name': 'WDBC Breast Cancer', 'short': 'wdbc'},
    {'id': 1471, 'name': 'EEG Eye State', 'short': 'eeg'},
]

datasets = {}
for cfg in DATASETS_CONFIG:
    print(f"Loading {cfg['name']} (OpenML ID={cfg['id']})...")
    d = fetch_openml(data_id=cfg['id'], as_frame=True, parser='auto', cache=True)

    X = d.data.copy()
    y = d.target.copy()

    # Encode labels to integers
    le = LabelEncoder()
    y = le.fit_transform(y.astype(str))

    # Force all features to numeric (some OpenML datasets have categorical cols)
    for col in X.columns:
        if X[col].dtype == 'object' or str(X[col].dtype).startswith('category'):
            X[col] = pd.to_numeric(X[col], errors='coerce')
    X = X.fillna(X.median())

    datasets[cfg['short']] = {
        'name': cfg['name'],
        'openml_id': cfg['id'],
        'X': X,
        'y': y,
        'n_samples': X.shape[0],
        'n_features': X.shape[1],
        'n_classes': len(np.unique(y)),
        'feature_names': list(X.columns),
        'class_labels': list(le.classes_),
    }

    counts = np.bincount(y)
    minority_pct = counts.min() / len(y) * 100
    print(f"  Shape: {X.shape}, classes: {len(np.unique(y))}, minority%: {minority_pct:.2f}%")
    print(f"  Class distribution: {dict(enumerate(counts))}")
    print()

print("All 3 datasets loaded successfully.")


Loading Blood Transfusion (OpenML ID=1464)...
  Shape: (748, 4), classes: 2, minority%: 23.80%
  Class distribution: {0: np.int64(570), 1: np.int64(178)}

Loading WDBC Breast Cancer (OpenML ID=1510)...
  Shape: (569, 30), classes: 2, minority%: 37.26%
  Class distribution: {0: np.int64(357), 1: np.int64(212)}

Loading EEG Eye State (OpenML ID=1471)...
  Shape: (14980, 14), classes: 2, minority%: 44.88%
  Class distribution: {0: np.int64(8257), 1: np.int64(6723)}

All 3 datasets loaded successfully.


# 5. Exploratory Data Analysis

We visualize the class distribution of each dataset. Mild imbalance is clearly visible across all three datasets.


In [ ]:
# EDA: Class distribution plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
colors = ['#2E86AB', '#A23B72']

for i, (key, ds) in enumerate(datasets.items()):
    counts = np.bincount(ds['y'])
    classes = [str(c) for c in range(len(counts))]
    bars = axes[i].bar(classes, counts, color=colors[:len(counts)], edgecolor='black', linewidth=0.5)
    axes[i].set_title(f"{ds['name']}\n(n={ds['n_samples']}, {ds['n_features']} feats, minority={counts.min()/len(ds['y'])*100:.1f}%)",
                       fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Class', fontsize=10)
    if i == 0:
        axes[i].set_ylabel('Count', fontsize=10)
    for bar, count in zip(bars, counts):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
                     f'{count}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Figure 1: Class Distribution Across Three Medical Datasets',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig1_class_distribution.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Figure 1 saved.")


Figure 1 saved.


In [ ]:
# EDA: Feature statistics summary table
print("=" * 90)
print("DATASET SUMMARY STATISTICS")
print("=" * 90)
summary_rows = []
for key, ds in datasets.items():
    counts = np.bincount(ds['y'])
    summary_rows.append({
        'Dataset': ds['name'],
        'OpenML_ID': ds['openml_id'],
        'Samples': ds['n_samples'],
        'Features': ds['n_features'],
        'Classes': ds['n_classes'],
        'Class_0_count': counts[0],
        'Class_1_count': counts[1] if len(counts) > 1 else 0,
        'Minority_%': round(counts.min() / ds['n_samples'] * 100, 2),
        'Mean_features': round(float(ds['X'].mean().mean()), 3),
        'Std_features': round(float(ds['X'].std().mean()), 3),
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(OUTPUT_DIR, 'dataset_summary.csv'), index=False)
print(f"\nSaved: {os.path.join(OUTPUT_DIR, 'dataset_summary.csv')}")


DATASET SUMMARY STATISTICS
           Dataset  OpenML_ID  Samples  Features  Classes  Class_0_count  Class_1_count  Minority_%  Mean_features  Std_features
 Blood Transfusion       1464      748         4        2            570            178       23.80        356.995       374.535
WDBC Breast Cancer       1510      569        30        2            357            212       37.26         61.891        34.905
     EEG Eye State       1471    14980        14        2           8257           6723       44.88       4316.882      1767.288

Saved: /content/cp_results/dataset_summary.csv


# 6. Methodology

## 6.1 Three-Way Data Split

For each dataset, we perform a **three-way stratified split**:
- **Training set (60%)** - used to fit the base classifier
- **Calibration set (20%)** - used to compute nonconformity score quantiles
- **Test set (20%)** - used to evaluate coverage and efficiency

This is the standard split conformal prediction protocol. Stratification preserves class proportions in each split.

## 6.2 Base Classifiers

We use **5 diverse base classifiers** to demonstrate CP's model-agnostic property:
1. **Logistic Regression** - linear baseline
2. **Random Forest** (100 trees) - bagging ensemble
3. **XGBoost** (100 estimators, max_depth=6) - gradient boosting
4. **LightGBM** (100 estimators, 31 leaves) - fast gradient boosting
5. **SVM (RBF kernel)** - non-parametric baseline

All classifiers use `predict_proba` to produce probability estimates; CP operates on these probabilities.

## 6.3 Conformal Prediction Methods

We implement **4 CP methods from scratch** using NumPy:

| Method | Score Function | Coverage Type | Key Property |
|--------|---------------|---------------|--------------|
| **Standard CP** | 1 - max_y pi_hat_y(x) | Marginal | Simple, calibrated on all classes |
| **Class-Conditional CP** | 1 - pi_hat_y(x), per-class quantile | Per-class | Fairness across classes |
| **Mondrian CP** | Per-partition quantile | Per-group | Generalization of CC (any partition) |
| **APS** | Cumulative softmax probability | Marginal + adaptive size | Adaptive set sizes |

## 6.4 Coverage Levels

We evaluate at **5 target coverage levels**: 1 - alpha in {0.80, 0.85, 0.90, 0.95, 0.99}.

## 6.5 Evaluation Metrics

| Metric | Definition |
|--------|-----------|
| **Empirical Coverage** | Fraction of test samples where true label is in prediction set |
| **Average Set Size** | Mean number of classes in prediction set (smaller = more efficient) |
| **Set Size Variance** | Variance of prediction set sizes (lower = more consistent) |
| **Class-Conditional Coverage Gap** | Max - min per-class coverage (lower = fairer) |
| **Point Accuracy** | Standard accuracy of the base classifier (argmax prediction) |
| **Brier Score** | Mean squared error of probability predictions (calibration) |
| **ECE** | Expected Calibration Error (10-bin discretization) |
| **Per-Class Efficiency** | Average set size conditioned on true class |

## 6.6 Total Configurations

5 base classifiers x 4 CP methods x 3 datasets x 5 coverage levels = **300 main configurations**,
plus cross-dataset transferability (6 configs) and reject-option analysis (15 configs).


## 6.7 Preprocessing and Data Splitting

In [ ]:
# Perform 3-way stratified split: 60% train / 20% calib / 20% test
def prepare_dataset_splits(ds, random_state=42):
    X = ds['X'].values
    y = ds['y']

    # 60% train, 40% temp (calib + test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.4, random_state=random_state, stratify=y
    )
    # Split 40% temp into 20% calib + 20% test
    X_calib, X_test, y_calib, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=random_state, stratify=y_temp
    )

    # Standardize features (fit on train only)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_calib_s = scaler.transform(X_calib)
    X_test_s = scaler.transform(X_test)

    return {
        'X_train': X_train_s, 'y_train': y_train,
        'X_calib': X_calib_s, 'y_calib': y_calib,
        'X_test': X_test_s, 'y_test': y_test,
        'scaler': scaler,
        'n_train': len(y_train), 'n_calib': len(y_calib), 'n_test': len(y_test),
    }

# Prepare splits for all datasets
splits = {}
for key, ds in datasets.items():
    splits[key] = prepare_dataset_splits(ds, random_state=RANDOM_STATE)
    s = splits[key]
    print(f"{ds['name']:30s} | Train: {s['n_train']:5d} | Calib: {s['n_calib']:5d} | Test: {s['n_test']:5d}")

print("\n3-way splits ready.")


Blood Transfusion              | Train:   448 | Calib:   150 | Test:   150
WDBC Breast Cancer             | Train:   341 | Calib:   114 | Test:   114
EEG Eye State                  | Train:  8988 | Calib:  2996 | Test:  2996

3-way splits ready.


## 6.8 Base Classifier Definitions

We define factory functions for each base classifier so they can be freshly re-instantiated for each dataset.


In [ ]:
def make_base_classifiers(random_state=42):
    # Return a dict of fresh base classifiers
    return {
        'LogReg': LogisticRegression(max_iter=2000, random_state=random_state),
        'RandomForest': RandomForestClassifier(n_estimators=100, random_state=random_state, n_jobs=-1),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=100, learning_rate=0.1, max_depth=6,
            use_label_encoder=False, eval_metric='logloss',
            random_state=random_state, n_jobs=-1, verbosity=0
        ),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=100, learning_rate=0.1, num_leaves=31,
            random_state=random_state, n_jobs=-1, verbose=-1
        ),
        'SVM_RBF': SVC(kernel='rbf', probability=True, random_state=random_state),
    }

# Quick sanity test: train all 5 on blood transfusion
s = splits['blood']
test_accuracies = {}
for name, model in make_base_classifiers().items():
    model.fit(s['X_train'], s['y_train'])
    y_pred = model.predict(s['X_test'])
    acc = accuracy_score(s['y_test'], y_pred)
    test_accuracies[name] = acc
    print(f"  {name:20s}: test_accuracy = {acc:.4f}")

print("\nAll 5 base classifiers trained successfully.")


  LogReg              : test_accuracy = 0.7867
  RandomForest        : test_accuracy = 0.7400
  XGBoost             : test_accuracy = 0.7667
  LightGBM            : test_accuracy = 0.7600
  SVM_RBF             : test_accuracy = 0.7667

All 5 base classifiers trained successfully.


# 7. Conformal Prediction Methods - From-Scratch Implementation

Below we implement all four CP methods using only NumPy. Each function returns:
- `pred_sets`: a binary indicator matrix of shape (n_test, n_classes) where 1 means "include class in set"
- `q_hat`: the computed quantile threshold (for transparency)
- `calib_scores`: the calibration nonconformity scores (for analysis)

The implementations are deliberately verbose and well-commented for pedagogical clarity.


## 7.1 Standard (Marginal) Conformal Prediction

Implements Theorem 1. Computes a single quantile over all calibration scores.


In [ ]:
# Standard (Marginal) Conformal Prediction
# Implements Theorem 1: marginal coverage guarantee.
# Nonconformity score: s(x, y) = 1 - p_hat_y(x) (LAC score).
def standard_cp(model, X_calib, y_calib, X_test, alpha):
    # Step 1: Compute calibration scores
    proba_calib = model.predict_proba(X_calib)
    # LAC score: 1 - probability of true class
    calib_scores = 1 - proba_calib[np.arange(len(y_calib)), y_calib]

    # Step 2: Compute (1-alpha) quantile with finite-sample correction
    n_calib = len(calib_scores)
    q_level = min(np.ceil((n_calib + 1) * (1 - alpha)) / n_calib, 1.0)
    q_hat = np.quantile(calib_scores, q_level, method='higher')

    # Step 3: Build prediction sets for test points
    proba_test = model.predict_proba(X_test)
    pred_sets = (proba_test >= (1 - q_hat)).astype(int)

    return pred_sets, q_hat, calib_scores

# Quick test
s = splits['blood']
model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
model.fit(s['X_train'], s['y_train'])
pred_sets, q_hat, _ = standard_cp(model, s['X_calib'], s['y_calib'], s['X_test'], alpha=0.05)
cov = (pred_sets[np.arange(len(s['y_test'])), s['y_test']] == 1).mean()
print(f"Standard CP test: alpha=0.05, q_hat={q_hat:.4f}, empirical_coverage={cov:.4f}, avg_set_size={pred_sets.sum(axis=1).mean():.4f}")


Standard CP test: alpha=0.05, q_hat=0.7109, empirical_coverage=0.9400, avg_set_size=1.3800


## 7.2 Class-Conditional Conformal Prediction

Implements Theorem 2. Computes a separate quantile for each class on the calibration scores restricted to that class.


In [ ]:
# Class-Conditional Conformal Prediction
# Implements Theorem 2: per-class coverage guarantee.
# Computes a separate quantile for each class.
def class_conditional_cp(model, X_calib, y_calib, X_test, alpha):
    proba_calib = model.predict_proba(X_calib)
    calib_scores = 1 - proba_calib[np.arange(len(y_calib)), y_calib]

    proba_test = model.predict_proba(X_test)
    n_classes = proba_test.shape[1]

    # Step 1: Compute per-class quantiles
    class_quantiles = np.zeros(n_classes)
    for k in range(n_classes):
        mask = (y_calib == k)
        if mask.sum() == 0:
            class_quantiles[k] = np.inf
            continue
        class_scores = calib_scores[mask]
        n_k = len(class_scores)
        q_level_k = min(np.ceil((n_k + 1) * (1 - alpha)) / n_k, 1.0)
        class_quantiles[k] = np.quantile(class_scores, q_level_k, method='higher')

    # Step 2: Build prediction sets using per-class thresholds
    pred_sets = np.zeros_like(proba_test, dtype=int)
    for k in range(n_classes):
        pred_sets[:, k] = (proba_test[:, k] >= (1 - class_quantiles[k])).astype(int)

    return pred_sets, class_quantiles, calib_scores

# Quick test
pred_sets, class_qs, _ = class_conditional_cp(model, s['X_calib'], s['y_calib'], s['X_test'], alpha=0.05)
print(f"Class-Conditional CP test: class_quantiles={class_qs}")
print(f"  Marginal coverage: {(pred_sets[np.arange(len(s['y_test'])), s['y_test']] == 1).mean():.4f}")
print(f"  Per-class coverage:")
for k in np.unique(s['y_test']):
    mask = s['y_test'] == k
    print(f"    Class {k}: {(pred_sets[mask, k] == 1).mean():.4f} (n={mask.sum()})")


Class-Conditional CP test: class_quantiles=[0.52426508 0.93392545]
  Marginal coverage: 0.9800
  Per-class coverage:
    Class 0: 1.0000 (n=115)
    Class 1: 0.9143 (n=35)


## 7.3 Mondrian Conformal Prediction

Implements Theorem 3. Generalization of Class-Conditional CP - allows arbitrary partitions. By default, we use class as the partition (which reduces to Class-Conditional CP). The function also supports custom partition functions.


In [ ]:
# Mondrian Conformal Prediction
# Implements Theorem 3: group-conditional coverage guarantee.
# Partition can be any function of (x, y) -> group_id.
# Default partition: y itself (equivalent to Class-Conditional CP).
def mondrian_cp(model, X_calib, y_calib, X_test, alpha, partition_fn=None):
    proba_calib = model.predict_proba(X_calib)
    calib_scores = 1 - proba_calib[np.arange(len(y_calib)), y_calib]

    # Default partition: by class label
    if partition_fn is None:
        calib_groups = y_calib.copy()
    else:
        calib_groups = np.array([partition_fn(x, y) for x, y in zip(X_calib, y_calib)])

    unique_groups = np.unique(calib_groups)
    group_quantiles = {}
    for g in unique_groups:
        mask = (calib_groups == g)
        if mask.sum() == 0:
            group_quantiles[g] = np.inf
            continue
        group_scores = calib_scores[mask]
        n_g = len(group_scores)
        q_level_g = min(np.ceil((n_g + 1) * (1 - alpha)) / n_g, 1.0)
        group_quantiles[g] = np.quantile(group_scores, q_level_g, method='higher')

    # Build prediction sets
    proba_test = model.predict_proba(X_test)
    n_classes = proba_test.shape[1]
    pred_sets = np.zeros_like(proba_test, dtype=int)

    if partition_fn is None:
        # Group = candidate class
        for k in range(n_classes):
            q_k = group_quantiles.get(k, np.inf)
            pred_sets[:, k] = (proba_test[:, k] >= (1 - q_k)).astype(int)
    else:
        # For each test point, consider each candidate class
        for i in range(len(X_test)):
            for k in range(n_classes):
                g = partition_fn(X_test[i], k)
                q_g = group_quantiles.get(g, np.inf)
                pred_sets[i, k] = int(proba_test[i, k] >= (1 - q_g))

    return pred_sets, group_quantiles, calib_scores

# Quick test (default = class partition = same as CC CP)
pred_sets, group_qs, _ = mondrian_cp(model, s['X_calib'], s['y_calib'], s['X_test'], alpha=0.05)
print(f"Mondrian CP test (class partition): group_quantiles={group_qs}")
print(f"  Coverage: {(pred_sets[np.arange(len(s['y_test'])), s['y_test']] == 1).mean():.4f}")


Mondrian CP test (class partition): group_quantiles={np.int64(0): np.float64(0.5242650808622014), np.int64(1): np.float64(0.9339254493752139)}
  Coverage: 0.9800


## 7.4 Adaptive Prediction Sets (APS)

Implements APS (Romano et al., 2020). Uses cumulative softmax probability as the nonconformity score, producing prediction sets whose sizes adapt to prediction difficulty.


In [ ]:
# Adaptive Prediction Sets (APS)
# Nonconformity score: cumulative sum of sorted (descending) softmax
# probabilities up to and including the true label.
# Produces adaptive-size prediction sets: small for confident predictions,
# large for ambiguous ones.
def aps_cp(model, X_calib, y_calib, X_test, alpha):
    proba_calib = model.predict_proba(X_calib)
    proba_test = model.predict_proba(X_test)
    n_calib = len(proba_calib)
    n_classes = proba_calib.shape[1]

    # Step 1: Compute calibration scores (cumulative softmax sum up to true label)
    calib_scores = np.zeros(n_calib)
    for i in range(n_calib):
        # Sort classes by descending probability
        order = np.argsort(-proba_calib[i])
        cumsum = 0.0
        for c in order:
            cumsum += proba_calib[i, c]
            if c == y_calib[i]:
                calib_scores[i] = cumsum
                break

    # Step 2: Compute quantile with finite-sample correction
    q_level = min(np.ceil((n_calib + 1) * (1 - alpha)) / n_calib, 1.0)
    q_hat = np.quantile(calib_scores, q_level, method='higher')

    # Step 3: Build adaptive prediction sets for test points
    n_test = len(proba_test)
    pred_sets = np.zeros((n_test, n_classes), dtype=int)
    for i in range(n_test):
        order = np.argsort(-proba_test[i])
        cumsum = 0.0
        for c in order:
            cumsum += proba_test[i, c]
            pred_sets[i, c] = 1
            if cumsum >= q_hat:
                break

    return pred_sets, q_hat, calib_scores

# Quick test
pred_sets, q_hat, _ = aps_cp(model, s['X_calib'], s['y_calib'], s['X_test'], alpha=0.05)
cov = (pred_sets[np.arange(len(s['y_test'])), s['y_test']] == 1).mean()
avg_size = pred_sets.sum(axis=1).mean()
print(f"APS test: q_hat={q_hat:.4f}, empirical_coverage={cov:.4f}, avg_set_size={avg_size:.4f}")


APS test: q_hat=1.0000, empirical_coverage=1.0000, avg_set_size=2.0000


# 8. Evaluation Metrics

We define a unified `compute_metrics` function that returns all 8 metrics for a given prediction set.


In [ ]:
# Compute all 8 evaluation metrics for a CP prediction set.
def compute_metrics(pred_sets, y_true, y_proba, point_predictions):
    n = len(y_true)
    n_classes = pred_sets.shape[1]
    set_sizes = pred_sets.sum(axis=1)

    # 1. Empirical coverage
    correct = pred_sets[np.arange(n), y_true] == 1
    empirical_coverage = float(correct.mean())

    # 2. Average set size
    avg_set_size = float(set_sizes.mean())

    # 3. Set size variance
    set_size_variance = float(set_sizes.var())

    # 4. Class-conditional coverage gap
    per_class_cov = []
    for k in range(n_classes):
        mask = (y_true == k)
        if mask.sum() > 0:
            per_class_cov.append(float((pred_sets[mask, k] == 1).mean()))
    class_cond_coverage_gap = float(max(per_class_cov) - min(per_class_cov)) if per_class_cov else 0.0

    # 5. Point accuracy (argmax of probabilities)
    point_accuracy = float((point_predictions == y_true).mean())

    # 6. Brier score (multi-class)
    onehot = np.zeros_like(y_proba)
    onehot[np.arange(n), y_true] = 1.0
    brier = float(np.mean(np.sum((y_proba - onehot) ** 2, axis=1)))

    # 7. Expected Calibration Error (10 bins, max over classes)
    n_bins = 10
    ece = 0.0
    for c in range(n_classes):
        proba_c = y_proba[:, c]
        true_c = (y_true == c).astype(int)
        for b in range(n_bins):
            lo, hi = b / n_bins, (b + 1) / n_bins
            mask = (proba_c >= lo) & (proba_c < hi)
            if mask.sum() > 0:
                ece += abs(proba_c[mask].mean() - true_c[mask].mean()) * mask.sum() / n

    # 8. Per-class efficiency
    per_class_efficiency = {}
    for k in range(n_classes):
        mask = (y_true == k)
        if mask.sum() > 0:
            per_class_efficiency[k] = float(set_sizes[mask].mean())

    return {
        'empirical_coverage': empirical_coverage,
        'avg_set_size': avg_set_size,
        'set_size_variance': set_size_variance,
        'class_cond_coverage_gap': class_cond_coverage_gap,
        'point_accuracy': point_accuracy,
        'brier_score': brier,
        'ece': float(ece),
        'per_class_efficiency': per_class_efficiency,
    }

# Test on the APS prediction set we just computed
proba_test = model.predict_proba(s['X_test'])
point_preds = model.predict(s['X_test'])
m = compute_metrics(pred_sets, s['y_test'], proba_test, point_preds)
print("Test metrics (APS, LogReg, blood, alpha=0.05):")
for k, v in m.items():
    print(f"  {k}: {v}")


Test metrics (APS, LogReg, blood, alpha=0.05):
  empirical_coverage: 1.0
  avg_set_size: 2.0
  set_size_variance: 0.0
  class_cond_coverage_gap: 0.0
  point_accuracy: 0.7866666666666666
  brier_score: 0.30338241072147665
  ece: 0.06404733422359768
  per_class_efficiency: {0: 2.0, 1: 2.0}


# 9. Main Experiments

We now run the full experiment matrix:
- **3 datasets** x **5 base classifiers** x **4 CP methods** x **5 coverage levels** = **300 configurations**

Each configuration produces 8 metrics. The full results table has 300 rows x ~12 columns.

**Estimated runtime on Google Colab free CPU:** ~25-35 minutes.

The progress bar shows which configuration is currently running.


In [ ]:
# Define experiment configuration
DATASET_KEYS = ['blood', 'wdbc', 'eeg']
MODEL_NAMES = ['LogReg', 'RandomForest', 'XGBoost', 'LightGBM', 'SVM_RBF']
CP_METHODS = ['Standard', 'ClassConditional', 'Mondrian', 'APS']
ALPHAS = [0.20, 0.15, 0.10, 0.05, 0.01]  # 80%, 85%, 90%, 95%, 99% coverage

# Function dispatcher
CP_FUNCTIONS = {
    'Standard': standard_cp,
    'ClassConditional': class_conditional_cp,
    'Mondrian': mondrian_cp,
    'APS': aps_cp,
}

print(f"Experiment matrix: {len(DATASET_KEYS)} datasets x {len(MODEL_NAMES)} models x {len(CP_METHODS)} CP methods x {len(ALPHAS)} alphas")
print(f"= {len(DATASET_KEYS) * len(MODEL_NAMES) * len(CP_METHODS) * len(ALPHAS)} total configurations")
print(f"Estimated runtime: ~25-35 minutes on Google Colab free CPU")
print()


Experiment matrix: 3 datasets x 5 models x 4 CP methods x 5 alphas
= 300 total configurations
Estimated runtime: ~25-35 minutes on Google Colab free CPU



In [ ]:
# ===== MAIN EXPERIMENT LOOP =====
all_results = []
start_time = time.time()
total_configs = len(DATASET_KEYS) * len(MODEL_NAMES) * len(CP_METHODS) * len(ALPHAS)
config_count = 0

for ds_key in DATASET_KEYS:
    ds = datasets[ds_key]
    s = splits[ds_key]
    print(f"\n{'='*70}")
    print(f"Dataset: {ds['name']} (n_train={s['n_train']}, n_calib={s['n_calib']}, n_test={s['n_test']})")
    print(f"{'='*70}")

    # Train all 5 base classifiers once per dataset
    trained_models = {}
    for model_name in MODEL_NAMES:
        model = make_base_classifiers(random_state=RANDOM_STATE)[model_name]
        model.fit(s['X_train'], s['y_train'])
        trained_models[model_name] = model
        print(f"  Trained {model_name} (test_accuracy = {accuracy_score(s['y_test'], model.predict(s['X_test'])):.4f})")

    # For each model + CP method + alpha
    for model_name in MODEL_NAMES:
        model = trained_models[model_name]
        proba_test = model.predict_proba(s['X_test'])
        point_preds = model.predict(s['X_test'])

        for cp_method in CP_METHODS:
            for alpha in ALPHAS:
                config_count += 1
                # Run CP method
                cp_fn = CP_FUNCTIONS[cp_method]
                try:
                    pred_sets, q_val, _ = cp_fn(model, s['X_calib'], s['y_calib'], s['X_test'], alpha)
                    # Compute metrics
                    metrics = compute_metrics(pred_sets, s['y_test'], proba_test, point_preds)
                except Exception as e:
                    print(f"  [ERROR] {ds_key}/{model_name}/{cp_method}/alpha={alpha}: {e}")
                    metrics = {k: np.nan for k in ['empirical_coverage','avg_set_size','set_size_variance',
                                                    'class_cond_coverage_gap','point_accuracy','brier_score','ece']}
                    metrics['per_class_efficiency'] = {}
                    q_val = np.nan

                # Record row
                row = {
                    'dataset': ds_key,
                    'dataset_name': ds['name'],
                    'openml_id': ds['openml_id'],
                    'model': model_name,
                    'cp_method': cp_method,
                    'alpha': alpha,
                    'target_coverage': 1 - alpha,
                }
                row.update({k: v for k, v in metrics.items() if not isinstance(v, dict)})
                row['per_class_efficiency'] = str(metrics.get('per_class_efficiency', {}))
                all_results.append(row)

                # Progress update every 25 configs
                if config_count % 25 == 0 or config_count == total_configs:
                    elapsed = time.time() - start_time
                    eta = (elapsed / config_count) * (total_configs - config_count)
                    print(f"  [{config_count:3d}/{total_configs}] {ds_key}/{model_name}/{cp_method}/alpha={alpha:.2f} | "
                          f"cov={metrics.get('empirical_coverage', 0):.3f} | "
                          f"elapsed={elapsed/60:.1f}m | ETA={eta/60:.1f}m")

elapsed = time.time() - start_time
print(f"\n{'='*70}")
print(f"COMPLETED: {total_configs} configurations in {elapsed/60:.1f} minutes")
print(f"{'='*70}")

# Convert to DataFrame
results_df = pd.DataFrame(all_results)
print(f"\nResults DataFrame shape: {results_df.shape}")
results_df.head(10)



Dataset: Blood Transfusion (n_train=448, n_calib=150, n_test=150)
  Trained LogReg (test_accuracy = 0.7867)
  Trained RandomForest (test_accuracy = 0.7400)
  Trained XGBoost (test_accuracy = 0.7667)
  Trained LightGBM (test_accuracy = 0.7600)
  Trained SVM_RBF (test_accuracy = 0.7667)
  [ 25/300] blood/RandomForest/Standard/alpha=0.01 | cov=1.000 | elapsed=0.0m | ETA=0.2m
  [ 50/300] blood/XGBoost/ClassConditional/alpha=0.01 | cov=0.967 | elapsed=0.0m | ETA=0.2m
  [ 75/300] blood/LightGBM/Mondrian/alpha=0.01 | cov=0.993 | elapsed=0.0m | ETA=0.1m
  [100/300] blood/SVM_RBF/APS/alpha=0.01 | cov=1.000 | elapsed=0.0m | ETA=0.1m

Dataset: WDBC Breast Cancer (n_train=341, n_calib=114, n_test=114)
  Trained LogReg (test_accuracy = 0.9649)
  Trained RandomForest (test_accuracy = 0.9825)
  Trained XGBoost (test_accuracy = 0.9737)
  Trained LightGBM (test_accuracy = 0.9737)
  Trained SVM_RBF (test_accuracy = 0.9825)
  [125/300] wdbc/RandomForest/Standard/alpha=0.01 | cov=0.991 | elapsed=0.1m | E

,dataset,dataset_name,openml_id,model,cp_method,alpha,target_coverage,empirical_coverage,avg_set_size,set_size_variance,class_cond_coverage_gap,point_accuracy,brier_score,ece,per_class_efficiency
0,blood,Blood Transfusion,1464,LogReg,Standard,0.20,0.80,0.800000,1.020000,0.019600,0.857143,0.786667,0.303382,0.064047,"{0: 1.017391304347826, 1: 1.0285714285714285}"
1,blood,Blood Transfusion,1464,LogReg,Standard,0.15,0.85,0.820000,1.086667,0.079156,0.771429,0.786667,0.303382,0.064047,"{0: 1.0695652173913044, 1: 1.1428571428571428}"
2,blood,Blood Transfusion,1464,LogReg,Standard,0.10,0.90,0.873333,1.226667,0.175289,0.542857,0.786667,0.303382,0.064047,"{0: 1.182608695652174, 1: 1.3714285714285714}"
3,blood,Blood Transfusion,1464,LogReg,Standard,0.05,0.95,0.940000,1.380000,0.235600,0.257143,0.786667,0.303382,0.064047,"{0: 1.2956521739130435, 1: 1.6571428571428573}"
4,blood,Blood Transfusion,1464,LogReg,Standard,0.01,0.99,0.980000,1.833333,0.138889,0.085714,0.786667,0.303382,0.064047,"{0: 1.817391304347826, 1: 1.8857142857142857}"
5,blood,Blood Transfusion,1464,LogReg,ClassConditional,0.20,0.80,0.780000,1.106667,0.095289,0.048447,0.786667,0.303382,0.064047,"{0: 1.0869565217391304, 1: 1.1714285714285715}"
6,blood,Blood Transfusion,1464,LogReg,ClassConditional,0.15,0.85,0.886667,1.500000,0.250000,0.001242,0.786667,0.303382,0.064047,"{0: 1.5043478260869565, 1: 1.4857142857142858}"
7,blood,Blood Transfusion,1464,LogReg,ClassConditional,0.10,0.90,0.926667,1.653333,0.226489,0.016149,0.786667,0.303382,0.064047,"{0: 1.6434782608695653, 1: 1.6857142857142857}"
8,blood,Blood Transfusion,1464,LogReg,ClassConditional,0.05,0.95,0.980000,1.813333,0.151822,0.085714,0.786667,0.303382,0.064047,"{0: 1.817391304347826, 1: 1.8}"
9,blood,Blood Transfusion,1464,LogReg,ClassConditional,0.01,0.99,0.980000,1.820000,0.147600,0.085714,0.786667,0.303382,0.064047,"{0: 1.817391304347826, 1: 1.8285714285714285}"


In [ ]:
# Save full results table to CSV
results_csv = os.path.join(OUTPUT_DIR, 'main_results.csv')
results_df.to_csv(results_csv, index=False)
print(f"Saved full results to: {results_csv}")
print(f"Total rows: {len(results_df)}")
print(f"\nColumn summary:")
print(results_df.dtypes)


Saved full results to: /content/cp_results/main_results.csv
Total rows: 300

Column summary:
dataset                     object
dataset_name                object
openml_id                    int64
model                       object
cp_method                   object
alpha                      float64
target_coverage            float64
empirical_coverage         float64
avg_set_size               float64
set_size_variance          float64
class_cond_coverage_gap    float64
point_accuracy             float64
brier_score                float64
ece                        float64
per_class_efficiency        object
dtype: object


# 10. Results Visualization

## 10.1 Coverage Validity Check

For valid CP, empirical coverage should be **>= target coverage** (theoretical guarantee).
The plot below shows empirical vs target coverage for each (model, CP method) combination, averaged across datasets.


In [ ]:
# Figure 2: Coverage Validity
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
cp_methods_list = ['Standard', 'ClassConditional', 'Mondrian', 'APS']
colors = {'LogReg': '#1f77b4', 'RandomForest': '#ff7f0e', 'XGBoost': '#2ca02c',
          'LightGBM': '#d62728', 'SVM_RBF': '#9467bd'}

for idx, cp_method in enumerate(cp_methods_list):
    ax = axes[idx]
    sub = results_df[results_df['cp_method'] == cp_method]
    for model_name in MODEL_NAMES:
        sub_m = sub[sub['model'] == model_name]
        # Average empirical coverage across datasets
        avg_cov = sub_m.groupby('target_coverage')['empirical_coverage'].mean()
        ax.plot(avg_cov.index, avg_cov.values, 'o-', label=model_name,
                color=colors[model_name], linewidth=2, markersize=7)
    ax.plot([0.7, 1.0], [0.7, 1.0], 'k--', alpha=0.5, label='Ideal')
    ax.set_xlabel('Target Coverage (1 - alpha)')
    if idx == 0:
        ax.set_ylabel('Empirical Coverage')
    ax.set_title(f'{cp_method} CP', fontweight='bold')
    ax.set_xlim([0.78, 1.01])
    ax.set_ylim([0.78, 1.01])
    ax.grid(True, alpha=0.3)
    if idx == 3:
        ax.legend(loc='lower right', fontsize=8)

plt.suptitle('Figure 2: Coverage Validity - Empirical vs Target Coverage (averaged across 3 datasets)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig2_coverage_validity.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Figure 2 saved.")


Figure 2 saved.


## 10.2 Coverage-Set Size Trade-off (Pareto Curves)

The fundamental trade-off in CP: higher target coverage requires larger prediction sets (less efficient). The Pareto-optimal method lies closest to the bottom-right corner (high coverage, small sets).


In [ ]:
# Figure 3: Coverage-Set Size Trade-off
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ds_keys_list = ['blood', 'wdbc', 'eeg']
ds_names_list = ['Blood Transfusion', 'WDBC Breast Cancer', 'EEG Eye State']
cp_colors = {'Standard': '#1f77b4', 'ClassConditional': '#ff7f0e',
             'Mondrian': '#2ca02c', 'APS': '#d62728'}
cp_markers = {'Standard': 'o', 'ClassConditional': 's', 'Mondrian': '^', 'APS': 'D'}

for idx, (ds_key, ds_name) in enumerate(zip(ds_keys_list, ds_names_list)):
    ax = axes[idx]
    sub = results_df[results_df['dataset'] == ds_key]
    # Average over models
    for cp_method in cp_methods_list:
        sub_cp = sub[sub['cp_method'] == cp_method]
        avg = sub_cp.groupby('target_coverage').agg(
            avg_cov=('empirical_coverage', 'mean'),
            avg_size=('avg_set_size', 'mean')
        ).reset_index()
        ax.plot(avg['avg_cov'], avg['avg_size'], cp_markers[cp_method] + '-',
                color=cp_colors[cp_method], label=cp_method, linewidth=2, markersize=8)
    ax.set_xlabel('Empirical Coverage')
    if idx == 0:
        ax.set_ylabel('Average Prediction Set Size')
    ax.set_title(f'{ds_name}', fontweight='bold')
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend(loc='upper left', fontsize=9)

plt.suptitle('Figure 3: Coverage-Set Size Trade-off (averaged over 5 base classifiers)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig3_coverage_setsize_tradeoff.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Figure 3 saved.")


Figure 3 saved.


## 10.3 Class-Conditional Coverage Gap

A lower gap means more **fair** CP - coverage is roughly equal across classes. Class-Conditional CP and Mondrian CP should achieve the smallest gaps.


In [ ]:
# Figure 4: Class-Conditional Coverage Gap Heatmap
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (ds_key, ds_name) in enumerate(zip(ds_keys_list, ds_names_list)):
    ax = axes[idx]
    sub = results_df[results_df['dataset'] == ds_key]
    # Pivot: rows=CP method, cols=model, values=mean coverage gap (averaged over alphas)
    pivot = sub.groupby(['cp_method', 'model'])['class_cond_coverage_gap'].mean().unstack()
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn_r',
                cbar_kws={'label': 'Coverage Gap'}, ax=ax, vmin=0, vmax=0.5,
                linewidths=0.5, linecolor='gray')
    ax.set_title(f'{ds_name}', fontweight='bold')
    ax.set_xlabel('Base Classifier')
    if idx == 0:
        ax.set_ylabel('CP Method')

plt.suptitle('Figure 4: Class-Conditional Coverage Gap (lower = fairer, averaged over coverage levels)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig4_class_conditional_gap.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Figure 4 saved.")


Figure 4 saved.


## 10.4 Calibration Analysis (Brier Score & ECE)

We compare the calibration quality of each base classifier. Lower Brier and ECE indicate better-calibrated probabilities, which directly impact CP prediction set quality.


In [ ]:
# Figure 5: Calibration Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Brier Score
brier_pivot = results_df.groupby(['dataset', 'model'])['brier_score'].mean().unstack()
brier_pivot.plot(kind='bar', ax=axes[0], edgecolor='black', linewidth=0.5)
axes[0].set_title('Brier Score by Dataset and Model', fontweight='bold')
axes[0].set_ylabel('Brier Score (lower = better)')
axes[0].set_xlabel('Dataset')
axes[0].legend(title='Model', fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')

# ECE
ece_pivot = results_df.groupby(['dataset', 'model'])['ece'].mean().unstack()
ece_pivot.plot(kind='bar', ax=axes[1], edgecolor='black', linewidth=0.5)
axes[1].set_title('Expected Calibration Error (ECE) by Dataset and Model', fontweight='bold')
axes[1].set_ylabel('ECE (lower = better)')
axes[1].set_xlabel('Dataset')
axes[1].legend(title='Model', fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Figure 5: Calibration Analysis of Base Classifiers', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig5_calibration.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Figure 5 saved.")


Figure 5 saved.


## 10.5 Detailed Results Tables

### Table 1: Main Results (Coverage = 95%, averaged across datasets)

In [ ]:
# Table 1: Main results at alpha=0.05 (95% coverage)
table1 = results_df[results_df['alpha'] == 0.05].groupby(['cp_method', 'model']).agg(
    empirical_coverage=('empirical_coverage', 'mean'),
    avg_set_size=('avg_set_size', 'mean'),
    class_cond_gap=('class_cond_coverage_gap', 'mean'),
    point_accuracy=('point_accuracy', 'mean'),
).reset_index()

# Pivot for readability
table1_pivot_cov = results_df[results_df['alpha'] == 0.05].pivot_table(
    index='cp_method', columns='model', values='empirical_coverage'
)
table1_pivot_size = results_df[results_df['alpha'] == 0.05].pivot_table(
    index='cp_method', columns='model', values='avg_set_size'
)

print("=" * 90)
print("TABLE 1: Empirical Coverage at 95% Target Coverage (averaged across 3 datasets)")
print("=" * 90)
print(table1_pivot_cov.round(4).to_string())
print()
print("=" * 90)
print("TABLE 1 (continued): Average Prediction Set Size at 95% Target Coverage")
print("=" * 90)
print(table1_pivot_size.round(4).to_string())

# Save tables
table1_pivot_cov.to_csv(os.path.join(OUTPUT_DIR, 'table1_coverage_95.csv'))
table1_pivot_size.to_csv(os.path.join(OUTPUT_DIR, 'table1_setsize_95.csv'))
print(f"\nSaved: table1_coverage_95.csv and table1_setsize_95.csv")


TABLE 1: Empirical Coverage at 95% Target Coverage (averaged across 3 datasets)
model             LightGBM  LogReg  RandomForest  SVM_RBF  XGBoost
cp_method                                                         
APS                 1.0000  1.0000        0.9933   1.0000   1.0000
ClassConditional    0.9721  0.9572        0.9603   0.9708   0.9599
Mondrian            0.9721  0.9572        0.9603   0.9708   0.9599
Standard            0.9556  0.9495        0.9647   0.9633   0.9565

TABLE 1 (continued): Average Prediction Set Size at 95% Target Coverage
model             LightGBM  LogReg  RandomForest  SVM_RBF  XGBoost
cp_method                                                         
APS                 2.0000  1.9999        1.8378   2.0000   2.0000
ClassConditional    1.3662  1.5237        1.3454   1.4344   1.3336
Mondrian            1.3662  1.5237        1.3454   1.4344   1.3336
Standard            1.2307  1.3928        1.2411   1.3166   1.2586

Saved: table1_coverage_95.csv and table1_s

### Table 2: Coverage-Set Size Trade-off (averaged across all models and datasets)

In [ ]:
# Table 2: Trade-off across coverage levels
table2 = results_df.groupby(['cp_method', 'target_coverage']).agg(
    empirical_coverage=('empirical_coverage', 'mean'),
    avg_set_size=('avg_set_size', 'mean'),
    class_cond_gap=('class_cond_coverage_gap', 'mean'),
).reset_index()

print("=" * 100)
print("TABLE 2: Coverage-Set Size Trade-off Across Coverage Levels")
print("=" * 100)
print(table2.round(4).to_string(index=False))

table2.to_csv(os.path.join(OUTPUT_DIR, 'table2_tradeoff.csv'), index=False)
print(f"\nSaved: table2_tradeoff.csv")


TABLE 2: Coverage-Set Size Trade-off Across Coverage Levels
       cp_method  target_coverage  empirical_coverage  avg_set_size  class_cond_gap
             APS             0.80              0.9987        1.9159          0.0057
             APS             0.85              0.9987        1.9346          0.0057
             APS             0.90              0.9987        1.9595          0.0057
             APS             0.95              0.9987        1.9675          0.0057
             APS             0.99              0.9987        1.9675          0.0057
ClassConditional             0.80              0.8021        1.0275          0.0799
ClassConditional             0.85              0.8624        1.1391          0.0633
ClassConditional             0.90              0.9121        1.2509          0.0543
ClassConditional             0.95              0.9641        1.4007          0.0221
ClassConditional             0.99              0.9885        1.5036          0.0214
        Mondrian

# 11. Cross-Dataset Transferability Analysis

A practically important question: **can a CP pipeline calibrated on one dataset transfer to a different dataset?**

We test this by:
1. Training the classifier on Dataset A
2. Calibrating CP on Dataset B
3. Testing on Dataset C

This evaluates robustness under distribution shift. Since the 3 datasets have different feature dimensions, we use the first min(n_features) features for all three to enable cross-dataset training.

We use RandomForest (robust ensemble) and Standard CP at 95% coverage.


In [ ]:
# Cross-dataset transferability
from itertools import permutations

transfer_results = []
alpha = 0.05
model_name = 'RandomForest'

print("Running cross-dataset transferability experiments...")
print(f"Model: {model_name} | CP Method: Standard | alpha: {alpha}")
print(f"6 permutations of (train, calibrate, test) across 3 datasets")
print("-" * 80)

for train_ds, calib_ds, test_ds in permutations(['blood', 'wdbc', 'eeg'], 3):
    s_train = splits[train_ds]
    s_calib = splits[calib_ds]
    s_test = splits[test_ds]

    n_features_train = s_train['X_train'].shape[1]
    n_features_calib = s_calib['X_calib'].shape[1]
    n_features_test = s_test['X_test'].shape[1]
    n_feats = min(n_features_train, n_features_calib, n_features_test)

    X_tr = s_train['X_train'][:, :n_feats]
    X_ca = s_calib['X_calib'][:, :n_feats]
    X_te = s_test['X_test'][:, :n_feats]
    # Use binary common subset
    y_tr = (s_train['y_train'] > 0).astype(int)
    y_ca = (s_calib['y_calib'] > 0).astype(int)
    y_te = (s_test['y_test'] > 0).astype(int)

    # Re-train on projected features
    model = make_base_classifiers(random_state=RANDOM_STATE)[model_name]
    model.fit(X_tr, y_tr)

    # Apply Standard CP
    try:
        pred_sets, q_hat, _ = standard_cp(model, X_ca, y_ca, X_te, alpha)
        proba_te = model.predict_proba(X_te)
        point_preds = model.predict(X_te)
        # Ensure pred_sets has same number of classes as y_te
        if pred_sets.shape[1] != len(np.unique(y_te)):
            n_classes_te = len(np.unique(y_te))
            pred_sets = pred_sets[:, :n_classes_te]
            proba_te = proba_te[:, :n_classes_te]
        metrics = compute_metrics(pred_sets, y_te, proba_te, point_preds)
    except Exception as e:
        print(f"  [ERROR] {train_ds}->{calib_ds}->{test_ds}: {e}")
        metrics = {'empirical_coverage': np.nan, 'avg_set_size': np.nan,
                   'class_cond_coverage_gap': np.nan, 'point_accuracy': np.nan,
                   'brier_score': np.nan, 'ece': np.nan, 'set_size_variance': np.nan,
                   'per_class_efficiency': {}}

    transfer_results.append({
        'train_ds': train_ds,
        'calib_ds': calib_ds,
        'test_ds': test_ds,
        'alpha': alpha,
        'empirical_coverage': metrics.get('empirical_coverage', np.nan),
        'avg_set_size': metrics.get('avg_set_size', np.nan),
        'class_cond_gap': metrics.get('class_cond_coverage_gap', np.nan),
        'point_accuracy': metrics.get('point_accuracy', np.nan),
    })
    print(f"  Train={train_ds:6s} -> Calib={calib_ds:6s} -> Test={test_ds:6s}: "
          f"cov={metrics.get('empirical_coverage', 0):.4f}, size={metrics.get('avg_set_size', 0):.4f}, "
          f"point_acc={metrics.get('point_accuracy', 0):.4f}")

transfer_df = pd.DataFrame(transfer_results)
transfer_df.to_csv(os.path.join(OUTPUT_DIR, 'cross_dataset_transfer.csv'), index=False)
print(f"\nSaved: cross_dataset_transfer.csv")
print(f"\nAverage empirical coverage across all 6 transfer configurations: "
      f"{transfer_df['empirical_coverage'].mean():.4f}")


Running cross-dataset transferability experiments...
Model: RandomForest | CP Method: Standard | alpha: 0.05
6 permutations of (train, calibrate, test) across 3 datasets
--------------------------------------------------------------------------------
  Train=blood  -> Calib=wdbc   -> Test=eeg   : cov=0.9950, size=1.9863, point_acc=0.5434
  Train=blood  -> Calib=eeg    -> Test=wdbc  : cov=0.8333, size=1.8246, point_acc=0.6228
  Train=wdbc   -> Calib=blood  -> Test=eeg   : cov=0.9489, size=1.8999, point_acc=0.5147
  Train=wdbc   -> Calib=eeg    -> Test=blood : cov=0.9733, size=1.9200, point_acc=0.6333
  Train=eeg    -> Calib=blood  -> Test=wdbc  : cov=0.9825, size=1.9561, point_acc=0.5965
  Train=eeg    -> Calib=wdbc   -> Test=blood : cov=0.8267, size=1.8133, point_acc=0.4133

Saved: cross_dataset_transfer.csv

Average empirical coverage across all 6 transfer configurations: 0.9266


# 12. Reject Option Integration

A natural application of CP is the **reject option**: abstain from prediction when the prediction set is ambiguous (contains multiple classes or no classes). We measure the **accuracy-coverage trade-off**: as we reject more, accuracy on remaining predictions should increase.

We use the size of the prediction set as the rejection criterion:
- If |C_hat(X)| = 1: accept the single-class prediction
- If |C_hat(X)| > 1: reject (ambiguous)
- If |C_hat(X)| = 0: reject (no class meets threshold)


In [ ]:
# Reject Option Analysis
reject_results = []
alphas_for_reject = [0.20, 0.15, 0.10, 0.05, 0.01]

print("Running reject option analysis...")
print(f"Model: RandomForest | CP Method: Standard")
print("-" * 80)

for ds_key in DATASET_KEYS:
    s = splits[ds_key]
    model = make_base_classifiers(random_state=RANDOM_STATE)['RandomForest']
    model.fit(s['X_train'], s['y_train'])

    for alpha in alphas_for_reject:
        pred_sets, _, _ = standard_cp(model, s['X_calib'], s['y_calib'], s['X_test'], alpha)
        set_sizes = pred_sets.sum(axis=1)

        # Accept only when set size == 1
        accept_mask = (set_sizes == 1)
        n_accept = accept_mask.sum()
        n_total = len(set_sizes)

        if n_accept > 0:
            accepted_preds = pred_sets[accept_mask].argmax(axis=1)
            accepted_acc = accuracy_score(s['y_test'][accept_mask], accepted_preds)
        else:
            accepted_acc = 0.0

        # Always-accept accuracy (baseline)
        always_preds = model.predict(s['X_test'])
        always_acc = accuracy_score(s['y_test'], always_preds)

        reject_results.append({
            'dataset': ds_key,
            'alpha': alpha,
            'target_coverage': 1 - alpha,
            'n_total': n_total,
            'n_accepted': n_accept,
            'accept_rate': n_accept / n_total,
            'reject_rate': 1 - n_accept / n_total,
            'accepted_accuracy': accepted_acc,
            'always_accept_accuracy': always_acc,
            'accuracy_gain': accepted_acc - always_acc,
        })

        print(f"  {ds_key:6s} | alpha={alpha:.2f} | accept_rate={n_accept/n_total:.3f} | "
              f"accepted_acc={accepted_acc:.4f} | baseline_acc={always_acc:.4f} | "
              f"gain={accepted_acc - always_acc:+.4f}")

reject_df = pd.DataFrame(reject_results)
reject_df.to_csv(os.path.join(OUTPUT_DIR, 'reject_option.csv'), index=False)
print(f"\nSaved: reject_option.csv")


Running reject option analysis...
Model: RandomForest | CP Method: Standard
--------------------------------------------------------------------------------
  blood  | alpha=0.20 | accept_rate=0.873 | accepted_acc=0.7634 | baseline_acc=0.7400 | gain=+0.0234
  blood  | alpha=0.15 | accept_rate=0.747 | accepted_acc=0.7857 | baseline_acc=0.7400 | gain=+0.0457
  blood  | alpha=0.10 | accept_rate=0.653 | accepted_acc=0.7959 | baseline_acc=0.7400 | gain=+0.0559
  blood  | alpha=0.05 | accept_rate=0.347 | accepted_acc=0.8846 | baseline_acc=0.7400 | gain=+0.1446
  blood  | alpha=0.01 | accept_rate=0.000 | accepted_acc=0.0000 | baseline_acc=0.7400 | gain=-0.7400
  wdbc   | alpha=0.20 | accept_rate=0.684 | accepted_acc=1.0000 | baseline_acc=0.9825 | gain=+0.0175
  wdbc   | alpha=0.15 | accept_rate=0.754 | accepted_acc=0.9884 | baseline_acc=0.9825 | gain=+0.0059
  wdbc   | alpha=0.10 | accept_rate=0.912 | accepted_acc=0.9904 | baseline_acc=0.9825 | gain=+0.0079
  wdbc   | alpha=0.05 | accept_rate

In [ ]:
# Figure 6: Reject Option Trade-off
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ds_keys_list = ['blood', 'wdbc', 'eeg']
ds_names_list = ['Blood Transfusion', 'WDBC Breast Cancer', 'EEG Eye State']

for idx, (ds_key, ds_name) in enumerate(zip(ds_keys_list, ds_names_list)):
    ax = axes[idx]
    sub = reject_df[reject_df['dataset'] == ds_key].sort_values('alpha')
    ax2 = ax.twinx()

    # Accept rate on left axis
    ax.plot(sub['alpha'], sub['accept_rate'], 'bo-', linewidth=2, markersize=8, label='Accept Rate')
    ax.set_xlabel('Alpha (1 - Target Coverage)')
    ax.set_ylabel('Accept Rate', color='blue')
    ax.tick_params(axis='y', labelcolor='blue')

    # Accepted accuracy on right axis
    ax2.plot(sub['alpha'], sub['accepted_accuracy'], 'rs-', linewidth=2, markersize=8, label='Accepted Accuracy')
    ax2.axhline(y=sub['always_accept_accuracy'].iloc[0], color='green', linestyle='--',
                label='Baseline (always accept)')
    ax2.set_ylabel('Accuracy', color='red')
    ax2.tick_params(axis='y', labelcolor='red')

    ax.set_title(f'{ds_name}', fontweight='bold')
    ax.grid(True, alpha=0.3)
    if idx == 0:
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=8)

plt.suptitle('Figure 6: Reject Option - Accept Rate vs Accepted Accuracy',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'fig6_reject_option.png'), dpi=300, bbox_inches='tight')
plt.show()
print("Figure 6 saved.")


Figure 6 saved.


# 13. Practitioner Decision Framework

Based on our experimental findings, we provide actionable guidelines for choosing among the 4 CP methods.

| Use Case | Recommended CP Method | Rationale |
|----------|----------------------|-----------|
| **Balanced classes, no fairness requirement** | Standard CP | Simplest, smallest sets |
| **Imbalanced classes, fairness required** | Class-Conditional CP | Per-class coverage guarantee |
| **Custom subgroups (e.g., demographics)** | Mondrian CP | Group-conditional coverage |
| **Adaptive set sizes, easy cases small sets** | APS | Adaptive to prediction difficulty |
| **High-stakes medical diagnosis** | Class-Conditional CP + Reject Option | Per-class fairness + abstention |


In [ ]:
# Generate decision framework from empirical findings
print("=" * 100)
print("PRACTITIONER DECISION FRAMEWORK (Derived from Empirical Results)")
print("=" * 100)

# For each CP method, summarize avg performance at 95% coverage
print("\n--- Performance Summary at 95% Coverage (averaged across datasets and models) ---")
summary_95 = results_df[results_df['alpha'] == 0.05].groupby('cp_method').agg(
    empirical_coverage=('empirical_coverage', 'mean'),
    avg_set_size=('avg_set_size', 'mean'),
    class_cond_gap=('class_cond_coverage_gap', 'mean'),
    coverage_target=('target_coverage', 'first'),
).reset_index()
print(summary_95.round(4).to_string(index=False))

print("\n--- Decision Rules ---")
print("1. If marginal coverage is sufficient and efficiency is priority:")
print(f"   -> Best avg set size: {summary_95.loc[summary_95['avg_set_size'].idxmin(), 'cp_method']} CP")
print()
print("2. If per-class fairness is required (low class-cond gap):")
print(f"   -> Lowest gap: {summary_95.loc[summary_95['class_cond_gap'].idxmin(), 'cp_method']} CP")
print()
print("3. If coverage validity is the strict requirement (coverage >= 0.95):")
best_cov = summary_95[summary_95['empirical_coverage'] >= 0.95]
if len(best_cov) > 0:
    print(f"   -> Methods meeting 0.95 marginal coverage: {list(best_cov['cp_method'])}")
else:
    print(f"   -> All methods have coverage below 0.95 (target was 0.95)")
    print(f"   -> Closest: {summary_95.loc[summary_95['empirical_coverage'].idxmax(), 'cp_method']} CP "
          f"({summary_95['empirical_coverage'].max():.4f})")

# Save framework
summary_95.to_csv(os.path.join(OUTPUT_DIR, 'practitioner_framework.csv'), index=False)
print(f"\nSaved: practitioner_framework.csv")


PRACTITIONER DECISION FRAMEWORK (Derived from Empirical Results)

--- Performance Summary at 95% Coverage (averaged across datasets and models) ---
       cp_method  empirical_coverage  avg_set_size  class_cond_gap  coverage_target
             APS              0.9987        1.9675          0.0057             0.95
ClassConditional              0.9641        1.4007          0.0221             0.95
        Mondrian              0.9641        1.4007          0.0221             0.95
        Standard              0.9579        1.2880          0.1031             0.95

--- Decision Rules ---
1. If marginal coverage is sufficient and efficiency is priority:
   -> Best avg set size: Standard CP

2. If per-class fairness is required (low class-cond gap):
   -> Lowest gap: APS CP

3. If coverage validity is the strict requirement (coverage >= 0.95):
   -> Methods meeting 0.95 marginal coverage: ['APS', 'ClassConditional', 'Mondrian', 'Standard']

Saved: practitioner_framework.csv


# 14. Summary and Final Outputs

## 14.1 Experiment Summary

This notebook executed **300 main configurations + 6 cross-dataset transfer + 15 reject-option experiments = ~321 total experiments**.

All results are saved to the output directory. The findings directly feed into the companion research paper (IEEE format, 12-14 pages).

## 14.2 Files Generated

The following files are saved in the output directory:

| File | Description |
|------|-------------|
| `main_results.csv` | Full 300-row results table (all metrics for all configurations) |
| `dataset_summary.csv` | Dataset characteristics summary |
| `table1_coverage_95.csv` | Coverage at 95% target (pivot table) |
| `table1_setsize_95.csv` | Set size at 95% target (pivot table) |
| `table2_tradeoff.csv` | Coverage-set size trade-off across all alpha levels |
| `cross_dataset_transfer.csv` | Cross-dataset transferability results |
| `reject_option.csv` | Reject option analysis results |
| `practitioner_framework.csv` | Decision framework summary |
| `figures/fig1_class_distribution.png` | Class distribution plot |
| `figures/fig2_coverage_validity.png` | Coverage validity plot |
| `figures/fig3_coverage_setsize_tradeoff.png` | Pareto trade-off plot |
| `figures/fig4_class_conditional_gap.png` | Class-cond gap heatmap |
| `figures/fig5_calibration.png` | Brier & ECE plots |
| `figures/fig6_reject_option.png` | Reject option plot |

## 14.3 For the Research Paper

After running this notebook:
1. Download all CSVs and figures from the output directory
2. Upload them to the research assistant chat
3. The assistant will write the full 12-14 page IEEE-format research paper using these results

---

## Author

**Muhammad Adeel**
- BBIT (Bachelor of Business Information Technology), 3rd Semester
- Virtual University of Pakistan, Lahore
- Email: ai.adeelv1@gmail.com
- LinkedIn: https://www.linkedin.com/in/muhammadadeelai/
- GitHub: https://github.com/adeeljames

## Acknowledgments

This work uses public datasets from OpenML (https://www.openml.org) and open-source Python libraries: scikit-learn, NumPy, pandas, matplotlib, seaborn, XGBoost, and LightGBM. The author thanks the open-source community for making this research possible without any financial cost.


In [ ]:
# Final summary: list all generated files
print("=" * 80)
print("ALL GENERATED FILES")
print("=" * 80)

import os
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = ' ' * 2 * (level + 1)
    for file in sorted(files):
        filepath = os.path.join(root, file)
        size_kb = os.path.getsize(filepath) / 1024
        print(f'{subindent}{file} ({size_kb:.1f} KB)')

print("\n" + "=" * 80)
print("NOTEBOOK EXECUTION COMPLETE")
print("=" * 80)
print(f"Author: Muhammad Adeel (BBIT, VU Pakistan, Lahore)")
print(f"Email: ai.adeelv1@gmail.com")
print(f"LinkedIn: https://www.linkedin.com/in/muhammadadeelai/")
print(f"GitHub: https://github.com/adeeljames")
print("=" * 80)
print("\nNext step: Download the CSV files and figures from the output directory")
print("and share them with the research assistant to write the full research paper.")


ALL GENERATED FILES
cp_results/
  cross_dataset_transfer.csv (0.6 KB)
  dataset_summary.csv (0.3 KB)
  main_results.csv (63.5 KB)
  practitioner_framework.csv (0.4 KB)
  reject_option.csv (1.7 KB)
  table1_coverage_95.csv (0.4 KB)
  table1_setsize_95.csv (0.4 KB)
  table2_tradeoff.csv (1.5 KB)
  figures/
    fig1_class_distribution.png (208.7 KB)
    fig2_coverage_validity.png (601.5 KB)
    fig3_coverage_setsize_tradeoff.png (311.7 KB)
    fig4_class_conditional_gap.png (393.7 KB)
    fig5_calibration.png (197.2 KB)
    fig6_reject_option.png (423.5 KB)

NOTEBOOK EXECUTION COMPLETE
Author: Muhammad Adeel (BBIT, VU Pakistan, Lahore)
Email: ai.adeelv1@gmail.com
LinkedIn: https://www.linkedin.com/in/muhammadadeelai/
GitHub: https://github.com/adeeljames

Next step: Download the CSV files and figures from the output directory
and share them with the research assistant to write the full research paper.
